In [2]:
import numpy as np
import matplotlib.pyplot as plt

In [21]:
# piston diameter [m]
d_M = 80/1000
# shot sleeve length [m]
l_m_activ = 756/1000
# total metal mass [t]
m_I = 4429/1e6
# overflow mass [t]
m_ov = 287/1e6
# part plus overflow mass [t]
m_A = 2985/1e6
# solid biscuit thickness [m]
t_biscuit_sol = 55/1000
# density solid [g/cm^3]
rho_s = 2.63
# density factor liquid/solid [-]
f_liq_sol = 0.92

In [22]:
# total metal volume liquid [m^3]
V_I_liq = m_I/(rho_s*f_liq_sol)
# total metal volume solid [m^3]
V_I_sol = m_I/rho_s
# shot sleeve cross section [m^2]
A_dM = np.square(d_M)*np.pi/4
# volume biscuit solid [m^3]
V_biscuit_sol = t_biscuit_sol*A_dM
# volume biscuit liquid [m^3]
V_biscuit_liq = V_biscuit_sol/f_liq_sol
# mold volume [m^3]
V_mold = V_I_sol - V_biscuit_sol
# part plus overflow volume solid [m^3]
V_A_sol = m_A/rho_s
# part without overflow volume solid [m^3]
V_part_sol = (m_A - m_ov)/rho_s
# runner volume mold [m^3]
V_runner = V_mold - V_A_sol
# shot sleeve volume [m^3]
V_chamber = A_dM*l_m_activ

In [23]:
# filling ratio shot sleeve liquid
fr = V_I_liq/V_chamber
# chamber filled
s_m_100 = l_m_activ*(1-fr)
# runner filled
s_ma = s_m_100 + V_runner/A_dM
# part filled = overflow reached
s_ov = s_ma + V_part_sol/A_dM
# all filled - still liquid
s_ffin = s_ma + V_A_sol/A_dM
# all filled - solid
s_Ifin_solid = s_ffin + (V_I_liq - V_I_sol)/A_dM

v_kr=(9.81*d_M)**0.5*(1.386-1.915*fr+0.616*fr**2) 

In [24]:
(1e3*s_m_100).round(), (1e3*s_ma).round(), (1e3*s_ov).round(), (1e3*s_ffin).round(), (1e3*s_Ifin_solid).round()

(np.float64(392.0),
 np.float64(446.0),
 np.float64(650.0),
 np.float64(672.0),
 np.float64(701.0))

In [28]:
def find_nearest_idx(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx
    
def get_time_points_discretized(s, v):  
    t = np.copy(s) * 0.  # Compute time from velocity and postition
    dt = (2 * (s - np.roll(s, 1)) / (v + np.roll(v,1)))[1:]    
    t[1:] = np.cumsum(dt)
    return t
    
def get_acceleration(v,t):
    return np.gradient(v, t)
    
def nogowizin(s, dm, lmactiv, fr, v_initial=0):
    v_kr=(9.81*dm)**0.5*(1.386-1.915*fr+0.616*fr**2)  # Eq. 5.13 Nogowizin
    a_kr=9.81*(dm/lmactiv)*(1+fr)/(1-fr)*(0.98-1.354*fr+0.436*fr**2)**2  # Eq. 5.18 Nogowizin
    s_kr=lmactiv*(1-fr)/(1+fr)  # Eq. 5.16 Nogowizin
      
    v = np.sqrt(np.square(v_initial) + s*2*a_kr)  # Array for piston velocity, Equation holds for constant acceleration
    v[v>v_kr] = v_kr
        
    t = get_time_points_discretized(s, v)

    t_kr = t[find_nearest_idx(s, s_kr)]
      
    return s, v, t, s_kr, v_kr, a_kr, t_kr
    
def buhler(s, dm, fr, s_m_100, vc, v_initial=0):
    # v_kr=(9.81*dm)**0.5*(1.386-1.915*fr+0.616*fr**2) * fraction_v_crit  # Eq. 5.13 Nogowizin
    v_kr = vc
    s_kr=s_m_100
    a_kr = np.square(v_kr) / (2*s_kr)  # Bühler: v_kr at sm100
          
    v = np.sqrt(np.square(v_initial) + s*2*a_kr)  # Array for piston velocity
    v[v>v_kr] = v_kr
      
    t = get_time_points_discretized(s, v)          
    t_kr = t[find_nearest_idx(s, s_kr)]
          
    return s, v, t, s_kr, v_kr, a_kr, t_kr
    
def standard_curve(s, s_vi1, v_i1, s_vi2, v_i2, v_initial=0):
    # Sector 0...1   
    dv_ds_1 = (v_i1 - v_initial) / s_vi1
    v = v_initial + dv_ds_1 * s
    v[v>v_i1] = v_i1
    
    # Sector 1...2
    if s_vi2 <= s_vi1:
        pass
    else:
        dv_ds_2 = (v_i2 - v_i1) / (s_vi2 - s_vi1)
        v[s>s_vi1] = v_i1 + dv_ds_2 * (s[s>s_vi1]-s_vi1)
        v[v>v_i2] = v_i2
    
    t = get_time_points_discretized(s, v)
    return s, v, t
     

In [29]:
   
cell_size = 1.e-4  # Cell size, resolution of shot sleeve
nr_of_cells = int(l_m_activ/cell_size)+1

v_init = 0.02

s = np.linspace(0,l_m_activ,nr_of_cells)


In [30]:
def point_line_distance(point, start, end):
    """Calculate the distance from a point to a line segment."""
    if np.all(start == end):
        return np.linalg.norm(point - start)
    else:
        return np.abs(np.linalg.norm(np.cross(end-start, start-point)) / np.linalg.norm(end-start))

def rdp(points, epsilon):
    """Recursive Ramer-Douglas-Peucker simplification."""
    dmax = 0.0
    index = 0
    start, end = points[0], points[-1]
    
    for i in range(1, len(points)-1):
        d = point_line_distance(points[i], start, end)
        if d > dmax:
            index = i
            dmax = d
    # If max distance is greater than epsilon, recursively simplify
    if dmax > epsilon:
        rec_results1 = rdp(points[:index+1], epsilon)
        rec_results2 = rdp(points[index:], epsilon)
        result = np.vstack((rec_results1[:-1], rec_results2))
    else:
        result = np.vstack((start, end))
    return result
    


In [31]:
def second_phase(s: np.ndarray, v: np.ndarray, opt: bool, s2: float, v2: float, s3: float, v3: float, s4: float, v4: float, sbrake: float, vbrake: float, s_ffin: float):

    index_s_vi3 = find_nearest_idx(s, s3)
    index_s_vi4 = find_nearest_idx(s, s4)
    index_s_break = find_nearest_idx(s, sbrake)
    index_sffin = find_nearest_idx(s, s_ffin)

    if opt:  # With first phase optimization
        index_s_vi25 = find_nearest_idx(s, s2)
        v_i25 = v[index_s_vi25]
        v[index_s_vi25:index_s_vi3] = v_i25 + (v3 - v_i25) / (s3 - s2) * (s[index_s_vi25:index_s_vi3]-s2)   
    else:  # Without first phase optimization
        index_s_vi2 = find_nearest_idx(s, s2)
        v[index_s_vi2:index_s_vi3] = v2 + (v3 - v2) / (s3 - s2) * (s[index_s_vi2:index_s_vi3]-s2)

    v[index_s_vi3:index_s_vi4] = v3 + (v4 - v3) / (s4 - s3) * (s[index_s_vi3:index_s_vi4]-s3)
    v[index_s_vi3] = v3

    v[v>v4] = v4
    v[index_s_vi4:] = v4

    v[index_s_break:index_sffin] = v4 + (vbrake - v4) / (s_ffin - sbrake) * (s[index_s_break:index_sffin]-sbrake)
    v[index_s_break] = v4
    v[index_sffin:] = vbrake

    t = get_time_points_discretized(s, v)

    return s, v, t

In [32]:
def curve_points(curve_type: str, s: np.ndarray, epsilon: float = 5e-3, *parameters) -> tuple[np.ndarray, np.ndarray]:
    """Generate shot curve points based on the specified curve type and parameters.

    Args:
        curve_type (str): First phase curve type ("Nogowizin", "Buhler any", "No optimization").
        s (np.ndarray): Stroke discretization array.
        epsilon (float, optional): Tolerance for RDP algorithm (resampling). Defaults to 5e-3.
        parameters (tuple): Parameters for the selected curve type. 
            - For "Nogowizin": (s2, s3, v3, s4, v4, sbrake, vbrake, s_ffin)
            - For "Buhler any": (v_crit, s2, s3, v3, s4, v4, sbrake, vbrake, s_ffin)
            - For "No optimization": (s1, v1, s2, v2, s3, v3, s4, v4, sbrake, vbrake, s_ffin)

    Returns:
        tuple[np.ndarray, np.ndarray]: t, v arrays after resampling.
    """
    if curve_type=="Nogowizin":
        s, v, t, _, v_kr, _, _ = nogowizin(s, d_M, l_m_activ, fr, v_initial=v_init)
        s, v, t = second_phase(s, v, True, parameters[0], v_kr, *parameters[1:])

        # resample based on s, v data
        points = np.column_stack((s, v))
        simplified_points = rdp(points, epsilon=epsilon)  # Adjust epsilon for desired tolerance

        s_simp, v_simp = simplified_points[:, 0], simplified_points[:, 1]
        t_simp = np.array([t[find_nearest_idx(s, s_simp_i)] for s_simp_i in s_simp])
    elif curve_type=="Buhler any":
        v_crit = parameters[0]
        s, v, t, _, _, _, _ = buhler(s, d_M, fr, s_m_100, v_crit, v_initial=v_init)
        s, v, t = second_phase(s, v, True, parameters[1], v_crit, *parameters[2:])
        # resample based on s, v data
        points = np.column_stack((s, v))
        simplified_points = rdp(points, epsilon=epsilon)  # Adjust epsilon for desired tolerance

        s_simp, v_simp = simplified_points[:, 0], simplified_points[:, 1]
        t_simp = np.array([t[find_nearest_idx(s, s_simp_i)] for s_simp_i in s_simp])
    elif curve_type=="No optimization":
        s_vi1, v_i1, s_vi2, v_i2 = parameters[:4]
        s, v, t = standard_curve(s, s_vi1, v_i1, s_vi2, v_i2, v_initial=v_init)
        s, v, t = second_phase(s, v, False, *parameters[2:])
        # resample based on t, v data
        points = np.column_stack((t, v))
        simplified_points = rdp(points, epsilon=epsilon)  # Adjust epsilon for desired tolerance

        t_simp, v_simp = simplified_points[:, 0], simplified_points[:, 1]
    else:
        v_simp = np.zeros_like(v)
        t_simp = np.zeros_like(t)
    return t_simp, v_simp


In [33]:
s2 = s_m_100
s3 = s_ma
v3 = 1.2
s4 = s3+0.02
v4 = 2.63
sbrake = s_ffin-0.02
vbrake = 1

In [34]:
curve_points("Nogowizin", s, 5e-3, s_m_100, s3, v3, s4, v4, sbrake, vbrake, s_ffin)

C:\Users\KoltzenburgNils\AppData\Local\Temp\ipykernel_37068\1205289587.py:6: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  return np.abs(np.linalg.norm(np.cross(end-start, start-point)) / np.linalg.norm(end-start))


(array([0.        , 0.1761581 , 0.35209223, 0.52829971, 0.7043667 ,
        0.98491729, 1.18988218, 1.25074403, 1.2617107 , 1.3312164 ,
        1.34308121, 1.37008121]),
 array([0.02      , 0.12986538, 0.23959107, 0.34948724, 0.4592958 ,
        0.63425501, 0.63411871, 1.2       , 2.63      , 2.63      ,
        1.        , 1.        ]))

In [35]:
v2 = 0.5*v_kr

In [36]:
curve_points("Buhler any", s, 5e-3, v2, s_m_100, s3, v3, s4, v4, sbrake, vbrake, s_ffin)

C:\Users\KoltzenburgNils\AppData\Local\Temp\ipykernel_37068\1205289587.py:6: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  return np.abs(np.linalg.norm(np.cross(end-start, start-point)) / np.linalg.norm(end-start))


(array([0.        , 0.33605451, 0.67149848, 1.34335089, 2.02300257,
        2.67770215, 2.75910748, 2.77007415, 2.83957986, 2.85144466,
        2.87844466]),
 array([0.02      , 0.0573684 , 0.09466891, 0.16937718, 0.2449527 ,
        0.31691481, 1.2       , 2.63      , 2.63      , 1.        ,
        1.        ]))

In [37]:
s1 = 0.16
v1 = 0.2
s2 = 0.399
v2 = 0.3
s3 = 0.54
v3 = 0.5
s4 = s3+0.02
v4 = 3.3
sbrake = s_ffin-0.019
vbrake = 0.5

In [38]:
curve_points("No optimization", s, 5e-3, s1, v1, s2, v2, s3, v3, s4, v4, sbrake, vbrake, s_ffin)

C:\Users\KoltzenburgNils\AppData\Local\Temp\ipykernel_37068\1205289587.py:6: DeprecationWarning: Arrays of 2-dimensional vectors are deprecated. Use arrays of 3-dimensional vectors instead. (deprecated in NumPy 2.0)
  return np.abs(np.linalg.norm(np.cross(end-start, start-point)) / np.linalg.norm(end-start))


(array([0.        , 0.70236392, 1.26015169, 1.68226289, 2.04674114,
        2.54743261, 3.01580275, 3.20345673, 3.37593481, 3.38941365,
        3.4348682 , 3.44767307, 3.50167307]),
 array([0.02      , 0.044075  , 0.08255   , 0.132725  , 0.2       ,
        0.24661088, 0.3       , 0.39148936, 0.5       , 3.3       ,
        3.3       , 0.5       , 0.5       ]))